# VATTR-EXP-001 — BoxCars Vehicle Attribute / Dimension Prior Classifier

Amaç: Araç crop'larından hız kestirimi için kullanılabilecek **araç tipi / görünüm / boyut ön bilgisi** üretmek.

Bu notebook sıfırdan ağır bir marka-model sistemi kurmak yerine, BoxCars116k üzerindeki hazır split/etiket yapısını kullanarak modern PyTorch tabanlı hafif bir classifier fine-tune eder. Çıktı doğrudan hız değeri değildir; `Speed Fusion Layer` için yardımcı sinyaldir.

Beklenen ana çıktı:

```text
vehicle_crop -> VATTR model -> body_type / fine_grained_label / dimension_prior / confidence
```

Önemli sınır:
- Bu modül tek başına mutlak km/s üretmez.
- Marka/model tahmini düşük güvenliyse wheelbase prior hız hesabında kullanılmaz.
- Hız kararı plate-scale, homography/track ve vehicle-dimension sinyallerinin fusion çıktısı olmalıdır.

## Senin Yapman Gerekenler

1. Google Drive içinde şu klasörü hazırla:

```text
/content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars/
```

2. BoxCars116k verisi için iki yol var. Varsayılan notebook önce resmi direct URL ile Drive'a indirmeyi dener:

```text
https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip
```

Drive hedefi:

```text
/content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars/BoxCars116k.zip
```

Drive içine ayrıca link notu bırakıldı:

```text
/content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars/BOXCARS116K_DOWNLOAD_LINKS.md
```

Resmi host DNS/erişim hatası verirse notebook Kaggle fallback kullanabilir. Bunun için Colab Secrets içinde `KAGGLE_USERNAME` ve `KAGGLE_KEY` tanımlı olmalı. Fallback dataset sayfası: https://www.kaggle.com/datasets/igorlashkov/boxcars-dataset

Alternatif olarak zip'i önceden açtıysan aynı klasör altında `dataset.pkl`, `classification_splits.pkl`, `atlas.pkl` veya `images/` yapısı bulunmalı.

3. İlk deneme için varsayılan `SMOKE_MODE=True` bırak. Ağır koşu için config hücresinde:

```python
SMOKE_MODE = False
MAX_VEHICLES_PER_SPLIT = None
MAX_INSTANCES_PER_VEHICLE = 4
EPOCHS = 20
BACKBONES = ['mobilenet_v3_large', 'efficientnet_b0']
```

4. Ham veri, checkpoint ve büyük çıktı dosyaları Git'e eklenmeyecek. Notebook kalıcı özetleri Drive'a yazar.

Kaynaklar:
- BoxCars GitHub: https://github.com/JakubSochor/BoxCars
- BoxCars116k dataset açıklaması: GitHub README içinde official dataset linki ve yapı tarifi bulunur.

In [1]:
# Cell 1 - Runtime setup + dependencies
import os
import sys
import json
import math
import time
import random
import shutil
import zipfile
import pickle
import subprocess
from pathlib import Path
from datetime import datetime, timezone


def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(packages))

# Colab çoğu torch paketini hazır getirir. Eksikleri kuruyoruz.
try:
    import torch
    import torchvision
except Exception:
    pip_install(['torch', 'torchvision'])

for module_name, packages in [
    ('pandas', ['pandas']),
    ('sklearn', ['scikit-learn']),
    ('PIL', ['pillow']),
    ('tqdm', ['tqdm']),
    ('cv2', ['opencv-python-headless']),
]:
    try:
        __import__(module_name)
    except Exception:
        pip_install(packages)

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
from torchvision import models

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA available: True
GPU: NVIDIA L4


In [2]:
# Cell 2 - Drive mount + config
from google.colab import drive

drive.mount('/content/drive')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

EXPERIMENT_ID = 'VATTR-EXP-001'
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
DRIVE_DATASET_ROOT = DRIVE_PROJECT_ROOT / 'datasets' / 'boxcars'
LOCAL_WORK_ROOT = Path('/content/anomali-road-safety-ai-work') / EXPERIMENT_ID
LOCAL_DATASET_ROOT = LOCAL_WORK_ROOT / 'datasets' / 'boxcars'
LOCAL_IMAGE_CACHE = LOCAL_WORK_ROOT / 'prepared_images'
LOCAL_RUN_ROOT = LOCAL_WORK_ROOT / 'runs'

DRIVE_RUN_ROOT = DRIVE_PROJECT_ROOT / 'runs' / 'vehicle_attribute' / EXPERIMENT_ID
DRIVE_MODEL_ROOT = DRIVE_PROJECT_ROOT / 'models' / 'checkpoints' / 'vehicle_attribute' / EXPERIMENT_ID
DRIVE_REPORT_ROOT = DRIVE_PROJECT_ROOT / 'reports' / 'vehicle_attribute' / EXPERIMENT_ID
DRIVE_METADATA_ROOT = DRIVE_PROJECT_ROOT / 'datasets' / 'boxcars_vehicle_attribute' / 'metadata'

for path in [DRIVE_DATASET_ROOT, LOCAL_WORK_ROOT, LOCAL_DATASET_ROOT, LOCAL_IMAGE_CACHE, LOCAL_RUN_ROOT, DRIVE_RUN_ROOT, DRIVE_MODEL_ROOT, DRIVE_REPORT_ROOT, DRIVE_METADATA_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

# Dataset acquisition. Varsayılan: Drive'a direct URL ile indir; DNS/erişim bozulursa Kaggle fallback denenebilir.
AUTO_DOWNLOAD_BOXCARS = True
DOWNLOAD_METHOD = 'direct'  # 'direct', 'kaggle', or 'manual'
ENABLE_KAGGLE_FALLBACK = True
BOXCARS_ZIP_NAME = 'BoxCars116k.zip'
KAGGLE_ARCHIVE_NAME = 'boxcars-dataset.zip'
KAGGLE_DATASET_SLUG = 'igorlashkov/boxcars-dataset'
BOXCARS_OFFICIAL_URLS = [
    'https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip',
]

# Target split selection. Dimension-prior için önce body split aranır; yoksa make/fine-grained split fallback olur.
PREFERRED_SPLIT_HINTS = ['body', 'make', 'medium', 'hard']

# Training mode. İlk koşu hızlı smoke test; ağır run için değerleri yükselt.
SMOKE_MODE = True
BACKBONES = ['mobilenet_v3_large']          # Heavy run: ['mobilenet_v3_large', 'efficientnet_b0']
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 3 if SMOKE_MODE else 20
LR = 3e-4
WEIGHT_DECAY = 1e-4
FREEZE_BACKBONE = True
MAX_VEHICLES_PER_SPLIT = 800 if SMOKE_MODE else None
MAX_INSTANCES_PER_VEHICLE = 2 if SMOKE_MODE else 4
NUM_WORKERS = 2
MIN_IMAGES_PER_CLASS = 5
USE_CLASS_WEIGHTS = True
USE_BALANCED_SAMPLER = True
CLASS_WEIGHT_POWER = 0.5  # sqrt inverse-frequency; less aggressive than full inverse.

# Optional smoke inference on existing crop folders after training.
SMOKE_CROP_DIRS = [
    DRIVE_PROJECT_ROOT / 'runs' / 'plate_ocr' / 'POCR-EXP-001-target-roi-crops',
]

print('DRIVE_DATASET_ROOT:', DRIVE_DATASET_ROOT)
print('LOCAL_DATASET_ROOT:', LOCAL_DATASET_ROOT)
print('DRIVE_MODEL_ROOT:', DRIVE_MODEL_ROOT)
print('SMOKE_MODE:', SMOKE_MODE)
print('BACKBONES:', BACKBONES)

Mounted at /content/drive
Device: cuda
DRIVE_DATASET_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars
LOCAL_DATASET_ROOT: /content/anomali-road-safety-ai-work/VATTR-EXP-001/datasets/boxcars
DRIVE_MODEL_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/vehicle_attribute/VATTR-EXP-001
SMOKE_MODE: True
BACKBONES: ['mobilenet_v3_large']


In [3]:
# Cell 3 - Dataset acquisition/preflight
import urllib.request


def run_cmd(cmd):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True, capture_output=False)




def configure_kaggle_credentials():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_json = kaggle_dir / 'kaggle.json'
    if kaggle_json.exists():
        os.chmod(kaggle_json, 0o600)
        return

    username = os.environ.get('KAGGLE_USERNAME')
    key = os.environ.get('KAGGLE_KEY')
    if not username or not key:
        try:
            from google.colab import userdata
            username = username or userdata.get('KAGGLE_USERNAME')
            key = key or userdata.get('KAGGLE_KEY')
        except Exception:
            pass

    if not username or not key:
        raise RuntimeError(
            'Kaggle fallback requires Colab Secrets KAGGLE_USERNAME and KAGGLE_KEY, '
            'or environment variables with the same names.'
        )

    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_text(json.dumps({'username': username, 'key': key}))
    os.chmod(kaggle_json, 0o600)


def download_boxcars_from_kaggle(target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    archive = target_dir / KAGGLE_ARCHIVE_NAME
    if archive.exists() and archive.stat().st_size > 10_000_000:
        print('Existing Kaggle archive found:', archive, 'size GB:', round(archive.stat().st_size / 1e9, 2))
        return archive
    pip_install(['kaggle'])
    configure_kaggle_credentials()
    cmd = [
        'kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET_SLUG,
        '-p', str(target_dir), '-f', KAGGLE_ARCHIVE_NAME, '--force'
    ]
    try:
        run_cmd(cmd)
    except Exception:
        # Some Kaggle mirrors do not support -f for the archive name; download full dataset archive instead.
        run_cmd(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET_SLUG, '-p', str(target_dir), '--force'])
    candidates = sorted(target_dir.glob('*.zip'), key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)
    if not candidates:
        raise FileNotFoundError('Kaggle download did not produce a zip archive under ' + str(target_dir))
    print('Kaggle archive selected:', candidates[0], 'size GB:', round(candidates[0].stat().st_size / 1e9, 2))
    return candidates[0]

def download_file_with_fallback(urls, target_path):
    target_path = Path(target_path)
    if target_path.exists() and target_path.stat().st_size > 10_000_000:
        print('Existing archive found:', target_path, 'size GB:', round(target_path.stat().st_size / 1e9, 2))
        return target_path
    errors = []
    for url in urls:
        print('Trying download:', url)
        part = target_path.with_suffix(target_path.suffix + '.part')
        try:
            urllib.request.urlretrieve(url, part)
            part.rename(target_path)
            print('Downloaded:', target_path)
            return target_path
        except Exception as exc:
            errors.append(f'urllib failed for {url}: {exc}')
            if part.exists():
                part.unlink()
        for tool in ['curl', 'wget']:
            try:
                if tool == 'curl':
                    run_cmd(['curl', '-L', '--retry', '5', '--retry-delay', '5', '-o', str(part), url])
                else:
                    run_cmd(['wget', '-O', str(part), '--tries=5', '--timeout=30', url])
                part.rename(target_path)
                print('Downloaded:', target_path)
                return target_path
            except Exception as exc:
                errors.append(f'{tool} failed for {url}: {exc}')
                if part.exists():
                    part.unlink()
    raise RuntimeError('Could not download BoxCars archive.\n' + '\n'.join(errors))


def find_named_file(root, names):
    root = Path(root)
    for name in names:
        direct = root / name
        if direct.exists():
            return direct
    for path in root.rglob('*'):
        if path.is_file() and path.name in names:
            return path
    return None



def extract_nested_archives_once(root, max_archives=5):
    """Extract likely nested BoxCars zip files once.
    Some mirrors wrap the official archive inside another dataset zip.
    """
    root = Path(root)
    archives = []
    for z in root.rglob('*.zip'):
        if z.name.startswith('.'):
            continue
        # Avoid repeatedly extracting the main source archive after it was already expanded.
        marker = z.with_suffix(z.suffix + '.extracted.marker')
        if marker.exists():
            continue
        archives.append(z)
    archives = sorted(archives, key=lambda x: x.stat().st_size if x.exists() else 0, reverse=True)[:max_archives]
    for archive in archives:
        out_dir = archive.parent / archive.stem
        out_dir.mkdir(parents=True, exist_ok=True)
        print('Extracting nested archive:', archive, '->', out_dir)
        try:
            with zipfile.ZipFile(archive, 'r') as zf:
                zf.extractall(out_dir)
            archive.with_suffix(archive.suffix + '.extracted.marker').write_text(datetime.now(timezone.utc).isoformat())
        except zipfile.BadZipFile:
            print('Skipping non-zip/bad archive:', archive)

def maybe_extract_boxcars():
    marker = LOCAL_DATASET_ROOT / '.boxcars_extracted.marker'
    if find_named_file(LOCAL_DATASET_ROOT, ['dataset.pkl']) and find_named_file(LOCAL_DATASET_ROOT, ['classification_splits.pkl']):
        print('Local extracted BoxCars files already exist.')
        return

    # If user placed extracted files in Drive, copy only metadata first; large image trees stay in Drive unless zip exists.
    if find_named_file(DRIVE_DATASET_ROOT, ['dataset.pkl']) and find_named_file(DRIVE_DATASET_ROOT, ['classification_splits.pkl']):
        print('Drive extracted BoxCars metadata detected. Using Drive as dataset root to avoid a huge copy.')
        return

    archive = DRIVE_DATASET_ROOT / BOXCARS_ZIP_NAME
    if not archive.exists() and AUTO_DOWNLOAD_BOXCARS and DOWNLOAD_METHOD != 'manual':
        if DOWNLOAD_METHOD == 'kaggle':
            archive = download_boxcars_from_kaggle(DRIVE_DATASET_ROOT)
        else:
            try:
                archive = download_file_with_fallback(BOXCARS_OFFICIAL_URLS, archive)
            except Exception as exc:
                print('Primary direct BoxCars download failed:', repr(exc))
                if ENABLE_KAGGLE_FALLBACK:
                    print('Trying Kaggle fallback:', KAGGLE_DATASET_SLUG)
                    archive = download_boxcars_from_kaggle(DRIVE_DATASET_ROOT)
                else:
                    raise

    if not archive.exists():
        raise FileNotFoundError(
            'BoxCars dataset archive was not found. Put it here:\n'
            f'  {DRIVE_DATASET_ROOT / BOXCARS_ZIP_NAME}\n'
            'or extract dataset.pkl/classification_splits.pkl under:\n'
            f'  {DRIVE_DATASET_ROOT}\n'
            'Official dataset link is documented by BoxCars GitHub: https://github.com/JakubSochor/BoxCars'
        )

    print('Extracting BoxCars archive to local runtime:', archive)
    with zipfile.ZipFile(archive, 'r') as zf:
        zf.extractall(LOCAL_DATASET_ROOT)
    extract_nested_archives_once(LOCAL_DATASET_ROOT)
    marker.write_text(datetime.now(timezone.utc).isoformat())
    print('Extraction complete:', LOCAL_DATASET_ROOT)

maybe_extract_boxcars()

DATASET_SEARCH_ROOTS = [LOCAL_DATASET_ROOT, DRIVE_DATASET_ROOT]
print('Dataset roots to inspect:')
for root in DATASET_SEARCH_ROOTS:
    print(' -', root, 'exists=', root.exists())
    if root.exists():
        for child in list(root.iterdir())[:12]:
            print('   ', 'dir ' if child.is_dir() else 'file', child.name)

Trying download: https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip
+ curl -L --retry 5 --retry-delay 5 -o /content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars/BoxCars116k.zip.part https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip
+ wget -O /content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars/BoxCars116k.zip.part --tries=5 --timeout=30 https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip
Primary direct BoxCars download failed: RuntimeError("Could not download BoxCars archive.\nurllib failed for https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip: <urlopen error [Errno -2] Name or service not known>\ncurl failed for https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip: Command '['curl', '-L', '--retry', '5', '--retry-delay', '5', '-o', '/content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars/BoxCars116k.zip.part', 'https://medusa.fit.vutbr.cz/traffic/data/BoxCars116k.zip']' returned non-zero exit status 6.\nwget failed for https://med

In [4]:
# Cell 4 - Load BoxCars metadata and inspect available splits

def load_pickle(path):
    path = Path(path)
    with path.open('rb') as f:
        try:
            return pickle.load(f)
        except UnicodeDecodeError:
            f.seek(0)
            return pickle.load(f, encoding='latin1')


def locate_dataset_files():
    found = {}
    for root in DATASET_SEARCH_ROOTS:
        if not root.exists():
            continue
        for key, names in {
            'dataset': ['dataset.pkl'],
            'splits': ['classification_splits.pkl'],
            'atlas': ['atlas.pkl'],
        }.items():
            if key not in found:
                path = find_named_file(root, names)
                if path:
                    found[key] = path
    return found

files = locate_dataset_files()
print('Located files:')
for key, value in files.items():
    print(' ', key, '->', value)

if 'dataset' not in files or 'splits' not in files:
    raise FileNotFoundError('dataset.pkl and classification_splits.pkl are required for VATTR-EXP-001.')

boxcars_dataset = load_pickle(files['dataset'])
classification_splits = load_pickle(files['splits'])
atlas = load_pickle(files['atlas']) if 'atlas' in files else None

print('\ndataset.pkl type:', type(boxcars_dataset))
if isinstance(boxcars_dataset, dict):
    print('dataset keys:', list(boxcars_dataset.keys())[:30])
print('classification_splits type:', type(classification_splits))
if isinstance(classification_splits, dict):
    print('split top-level keys:', list(classification_splits.keys())[:30])
print('atlas available:', atlas is not None)

Located files:
  dataset -> /content/anomali-road-safety-ai-work/VATTR-EXP-001/datasets/boxcars/BoxCars116k/dataset.pkl
  splits -> /content/anomali-road-safety-ai-work/VATTR-EXP-001/datasets/boxcars/BoxCars116k/classification_splits.pkl
  atlas -> /content/anomali-road-safety-ai-work/VATTR-EXP-001/datasets/boxcars/BoxCars116k/atlas.pkl

dataset.pkl type: <class 'dict'>
dataset keys: ['cameras', 'samples']
classification_splits type: <class 'dict'>
split top-level keys: ['body', 'medium', 'hard', 'make']
atlas available: True


In [5]:
# Cell 5 - Select split and build image-backed training records

def as_text(value):
    if isinstance(value, bytes):
        return value.decode('utf-8', errors='ignore')
    return str(value)


def find_split_candidates(splits):
    candidates = []
    if isinstance(splits, dict):
        for key, value in splits.items():
            if isinstance(value, dict) and any(k in value for k in ['train', 'training', 'validation', 'val', 'test']):
                candidates.append((as_text(key), value))
            elif isinstance(value, dict):
                for subkey, subval in value.items():
                    if isinstance(subval, dict) and any(k in subval for k in ['train', 'training', 'validation', 'val', 'test']):
                        candidates.append((as_text(key) + '/' + as_text(subkey), subval))
    return candidates

split_candidates = find_split_candidates(classification_splits)
print('Split candidates:')
for name, split_obj in split_candidates:
    print(' -', name, 'keys=', list(split_obj.keys())[:20])

if not split_candidates:
    raise ValueError('No train/validation/test split candidate was found in classification_splits.pkl')

selected_name, selected_split = None, None
for hint in PREFERRED_SPLIT_HINTS:
    for name, split_obj in split_candidates:
        if hint.lower() in name.lower():
            selected_name, selected_split = name, split_obj
            break
    if selected_split is not None:
        break
if selected_split is None:
    selected_name, selected_split = split_candidates[0]

print('\nSelected split:', selected_name)
print('Selected keys:', selected_split.keys())


def get_split_entries(split_obj, split_name):
    aliases = {
        'train': ['train', 'training'],
        'val': ['validation', 'val', 'valid'],
        'test': ['test'],
    }[split_name]
    for key in aliases:
        if key in split_obj:
            return list(split_obj[key])
    return []


def reverse_mapping(split_obj):
    mapping = split_obj.get('types_mapping') or split_obj.get('type_mapping') or split_obj.get('classes') or split_obj.get('mapping')
    if isinstance(mapping, dict):
        # Often textual label -> class id.
        rev = {}
        for k, v in mapping.items():
            try:
                rev[int(v)] = as_text(k)
            except Exception:
                try:
                    rev[int(k)] = as_text(v)
                except Exception:
                    pass
        return rev
    if isinstance(mapping, (list, tuple)):
        return {idx: as_text(v) for idx, v in enumerate(mapping)}
    return {}

class_id_to_name = reverse_mapping(selected_split)
print('Class map sample:', list(class_id_to_name.items())[:20])

samples = boxcars_dataset.get('samples') if isinstance(boxcars_dataset, dict) else None
if samples is None:
    raise ValueError('dataset.pkl does not contain samples; unsupported BoxCars format.')

# Index image files. This is used if image tree exists. Atlas fallback is used otherwise.
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
image_index = {}
for root in DATASET_SEARCH_ROOTS:
    if not root.exists():
        continue
    image_root_candidates = [p for p in root.rglob('images') if p.is_dir()]
    for image_root in image_root_candidates:
        print('Indexing images under:', image_root)
        for img in image_root.rglob('*'):
            if img.suffix.lower() in IMAGE_EXTS:
                stem = img.stem
                image_index.setdefault(stem, img)
                image_index.setdefault(img.name, img)
print('Indexed image keys:', len(image_index))


def get_sample(vehicle_id):
    try:
        if isinstance(samples, list):
            return samples[int(vehicle_id)]
        if isinstance(samples, dict):
            return samples.get(vehicle_id) or samples.get(str(vehicle_id)) or samples.get(int(vehicle_id))
    except Exception:
        return None
    return None


def instance_candidates(vehicle_id, instance_id):
    v = as_text(vehicle_id)
    i = as_text(instance_id)
    return [
        f'{v}_{i}', f'{v}-{i}', f'{v}/{i}', f'{v}_{i}.jpg', f'{v}-{i}.jpg', i, f'{i}.jpg'
    ]


def find_image_path(vehicle_id, instance_id, inst):
    for key in ['image', 'image_path', 'path', 'filename', 'file_name', 'name']:
        if isinstance(inst, dict) and key in inst:
            raw = as_text(inst[key])
            p = Path(raw)
            candidates = [p]
            for root in DATASET_SEARCH_ROOTS:
                candidates.append(root / raw)
                candidates.append(root / 'images' / raw)
            for cand in candidates:
                if cand.exists() and cand.is_file():
                    return cand
            if p.name in image_index:
                return image_index[p.name]
            if p.stem in image_index:
                return image_index[p.stem]
    for cand in instance_candidates(vehicle_id, instance_id):
        if cand in image_index:
            return image_index[cand]
        stem = Path(cand).stem
        if stem in image_index:
            return image_index[stem]
    return None


def decode_atlas_image(vehicle_id, instance_id):
    if atlas is None:
        return None
    keys_v = [vehicle_id, str(vehicle_id), int(vehicle_id) if str(vehicle_id).isdigit() else vehicle_id]
    keys_i = [instance_id, str(instance_id), int(instance_id) if str(instance_id).isdigit() else instance_id]
    for kv in keys_v:
        try:
            vehicle_blob = atlas[kv]
        except Exception:
            continue
        for ki in keys_i:
            try:
                encoded = vehicle_blob[ki]
                arr = np.frombuffer(encoded, dtype=np.uint8)
                import cv2
                img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
                if img is not None:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    return Image.fromarray(img)
            except Exception:
                continue
    return None


def materialize_record_image(vehicle_id, instance_id, inst, split_name, row_idx):
    image_path = find_image_path(vehicle_id, instance_id, inst)
    if image_path:
        return str(image_path), 'file'
    img = decode_atlas_image(vehicle_id, instance_id)
    if img is not None:
        out_dir = LOCAL_IMAGE_CACHE / split_name
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{vehicle_id}_{instance_id}_{row_idx}.jpg'
        if not out_path.exists():
            img.save(out_path, quality=92)
        return str(out_path), 'atlas'
    return None, 'missing'


def parse_split_tuple(entry):
    # Expected BoxCars entry: (vehicle_id, class_id). Keep robust fallbacks.
    if isinstance(entry, dict):
        vehicle_id = entry.get('vehicle_id') or entry.get('id') or entry.get('sample_id')
        class_id = entry.get('class_id') or entry.get('label') or entry.get('type_id')
        return vehicle_id, class_id
    if isinstance(entry, (list, tuple)) and len(entry) >= 2:
        return entry[0], entry[1]
    raise ValueError(f'Unsupported split entry: {entry}')


def list_instances(sample):
    if not isinstance(sample, dict):
        return []
    insts = sample.get('instances') or sample.get('imgs') or sample.get('images') or []
    if isinstance(insts, dict):
        out = []
        for k, v in insts.items():
            if isinstance(v, dict):
                vv = dict(v)
                vv.setdefault('instance_id', k)
                out.append(vv)
            else:
                out.append({'instance_id': k, 'value': v})
        return out
    return list(insts)

records = []
missing_images = 0
missing_samples = 0
for split_name in ['train', 'val', 'test']:
    entries = get_split_entries(selected_split, split_name)
    if MAX_VEHICLES_PER_SPLIT is not None:
        entries = entries[:MAX_VEHICLES_PER_SPLIT]
    print(f'Building records for {split_name}: vehicles={len(entries)}')
    for row_idx, entry in enumerate(tqdm(entries, desc=f'{split_name} vehicles')):
        vehicle_id, class_id = parse_split_tuple(entry)
        sample = get_sample(vehicle_id)
        if sample is None:
            missing_samples += 1
            continue
        label_id = int(class_id)
        label_name = class_id_to_name.get(label_id, f'class_{label_id}')
        insts = list_instances(sample)
        if MAX_INSTANCES_PER_VEHICLE is not None:
            insts = insts[:MAX_INSTANCES_PER_VEHICLE]
        for inst_idx, inst in enumerate(insts):
            if not isinstance(inst, dict):
                inst = {'instance_id': inst_idx, 'value': inst}
            instance_id = inst.get('instance_id') or inst.get('id') or inst.get('image_id') or inst_idx
            image_path, image_source = materialize_record_image(vehicle_id, instance_id, inst, split_name, f'{row_idx}_{inst_idx}')
            if image_path is None:
                missing_images += 1
                continue
            records.append({
                'split': split_name,
                'vehicle_id': vehicle_id,
                'instance_id': instance_id,
                'label_id_source': label_id,
                'label_name': label_name,
                'image_path': image_path,
                'image_source': image_source,
                'to_camera': sample.get('to_camera') if isinstance(sample, dict) else None,
                'selected_split': selected_name,
            })

records_df = pd.DataFrame(records)
print('records:', records_df.shape)
print('missing_samples:', missing_samples, 'missing_images:', missing_images)
if records_df.empty:
    raise RuntimeError('No image-backed records were produced. Check BoxCars image tree or atlas.pkl availability.')

# Drop classes too small for meaningful validation.
counts = records_df[records_df['split'] == 'train']['label_name'].value_counts()
valid_labels = set(counts[counts >= MIN_IMAGES_PER_CLASS].index)
records_df = records_df[records_df['label_name'].isin(valid_labels)].reset_index(drop=True)
label_names = sorted(records_df['label_name'].unique())
label_to_id = {name: idx for idx, name in enumerate(label_names)}
records_df['label_id'] = records_df['label_name'].map(label_to_id)

metadata_csv = DRIVE_METADATA_ROOT / f'{EXPERIMENT_ID.lower()}_records.csv'
records_df.to_csv(metadata_csv, index=False)
print('Filtered records:', records_df.shape)
print('Classes:', len(label_names))
print('metadata_csv:', metadata_csv)
display(records_df.groupby(['split']).size().to_frame('images'))
display(records_df.groupby(['split', 'label_name']).size().reset_index(name='images').head(20))

Split candidates:
 - body keys= ['test', 'types_mapping', 'train', 'test_cameras', 'validation']
 - medium keys= ['test', 'types_mapping', 'train', 'test_cameras', 'validation']
 - hard keys= ['test', 'types_mapping', 'train', 'test_cameras', 'validation']
 - make keys= ['test', 'types_mapping', 'train', 'test_cameras', 'validation']

Selected split: body
Selected keys: dict_keys(['test', 'types_mapping', 'train', 'test_cameras', 'validation'])
Class map sample: [(4, 'van'), (0, 'combi'), (5, 'mpv'), (1, 'suv'), (3, 'sedan'), (2, 'hatchback')]
Indexing images under: /content/anomali-road-safety-ai-work/VATTR-EXP-001/datasets/boxcars/BoxCars116k/images
Indexed image keys: 465144
Building records for train: vehicles=800


train vehicles:   0%|          | 0/800 [00:00<?, ?it/s]

Building records for val: vehicles=771


val vehicles:   0%|          | 0/771 [00:00<?, ?it/s]

Building records for test: vehicles=800


test vehicles:   0%|          | 0/800 [00:00<?, ?it/s]

records: (4742, 9)
missing_samples: 0 missing_images: 0
Filtered records: (4742, 10)
Classes: 6
metadata_csv: /content/drive/MyDrive/anomali-road-safety-ai/datasets/boxcars_vehicle_attribute/metadata/vattr-exp-001_records.csv


,images
split,
test,1600
train,1600
val,1542


,split,label_name,images
0,test,combi,722
1,test,hatchback,584
2,test,mpv,14
3,test,sedan,174
4,test,suv,42
5,test,van,64
6,train,combi,790
7,train,hatchback,334
8,train,mpv,22
9,train,sedan,358


In [6]:
# Cell 6 - Dataset/DataLoader helpers
class VehicleAttributeDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row.image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = int(row.label_id)
        return image, label

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.08, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = records_df[records_df['split'] == 'train'].copy()
val_df = records_df[records_df['split'] == 'val'].copy()
test_df = records_df[records_df['split'] == 'test'].copy()

if val_df.empty:
    print('Validation split empty; making deterministic split from train for smoke purposes.')
    val_df = train_df.sample(frac=0.15, random_state=SEED)
    train_df = train_df.drop(val_df.index)
if test_df.empty:
    print('Test split empty; making deterministic split from remaining train for smoke purposes.')
    test_df = train_df.sample(frac=0.10, random_state=SEED + 1)
    train_df = train_df.drop(test_df.index)

num_classes = len(label_names)
train_counts = train_df['label_id'].value_counts().reindex(range(num_classes), fill_value=0).sort_index()
if (train_counts == 0).any():
    missing = [label_names[i] for i, c in train_counts.items() if c == 0]
    raise ValueError('Train split has classes with zero images: ' + ', '.join(missing))

class_weights_np = (len(train_df) / (num_classes * train_counts.to_numpy(dtype=float))) ** CLASS_WEIGHT_POWER
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
print('train class counts:', {label_names[i]: int(c) for i, c in train_counts.items()})
print('class weights:', {label_names[i]: round(float(w), 4) for i, w in enumerate(class_weights_np)})

train_dataset = VehicleAttributeDataset(train_df, train_tf)
val_dataset = VehicleAttributeDataset(val_df, eval_tf)
test_dataset = VehicleAttributeDataset(test_df, eval_tf)

if USE_BALANCED_SAMPLER:
    sample_weights = train_df['label_id'].map(lambda idx: float(class_weights_np[int(idx)])).to_numpy()
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), persistent_workers=NUM_WORKERS > 0)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), persistent_workers=NUM_WORKERS > 0)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), persistent_workers=NUM_WORKERS > 0)

print('num_classes:', num_classes)
print('train/val/test images:', len(train_df), len(val_df), len(test_df))
print('USE_CLASS_WEIGHTS:', USE_CLASS_WEIGHTS, 'USE_BALANCED_SAMPLER:', USE_BALANCED_SAMPLER)

train class counts: {'combi': 790, 'hatchback': 334, 'mpv': 22, 'sedan': 358, 'suv': 58, 'van': 38}
class weights: {'combi': 0.581, 'hatchback': 0.8935, 'mpv': 3.4816, 'sedan': 0.8631, 'suv': 2.1442, 'van': 2.6491}
num_classes: 6
train/val/test images: 1600 1542 1600
USE_CLASS_WEIGHTS: True USE_BALANCED_SAMPLER: True


In [7]:
# Cell 7 - Model factory + training loop

def create_model(backbone_name, num_classes):
    if backbone_name == 'mobilenet_v3_large':
        weights = models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
        model = models.mobilenet_v3_large(weights=weights)
        in_features = model.classifier[-1].in_features
        if FREEZE_BACKBONE:
            for p in model.features.parameters():
                p.requires_grad = False
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    if backbone_name == 'mobilenet_v3_small':
        weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[-1].in_features
        if FREEZE_BACKBONE:
            for p in model.features.parameters():
                p.requires_grad = False
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    if backbone_name == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[-1].in_features
        if FREEZE_BACKBONE:
            for p in model.features.parameters():
                p.requires_grad = False
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    if backbone_name == 'convnext_tiny':
        weights = models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1
        model = models.convnext_tiny(weights=weights)
        in_features = model.classifier[-1].in_features
        if FREEZE_BACKBONE:
            for p in model.features.parameters():
                p.requires_grad = False
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    raise ValueError('Unsupported backbone: ' + backbone_name)


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    preds, labels = [], []
    criterion = nn.CrossEntropyLoss(weight=class_weights if USE_CLASS_WEIGHTS else None)
    for images, y in tqdm(loader, leave=False):
        images = images.to(device)
        y = y.to(device)
        with torch.set_grad_enabled(is_train):
            logits = model(images)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total_loss += float(loss.item()) * images.size(0)
        preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
        labels.extend(y.detach().cpu().numpy().tolist())
    avg_loss = total_loss / max(1, len(loader.dataset))
    acc = accuracy_score(labels, preds) if labels else 0.0
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0) if labels else 0.0
    return {'loss': avg_loss, 'accuracy': acc, 'macro_f1': macro_f1, 'preds': preds, 'labels': labels}

all_histories = []
best_model_info = None

for backbone in BACKBONES:
    print('\n=== Training', backbone, '===')
    model = create_model(backbone, num_classes).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
    best_val_f1 = -1.0
    best_path = LOCAL_RUN_ROOT / f'{EXPERIMENT_ID}-{backbone}-best.pth'
    history = []
    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader, None)
        row = {
            'experiment_id': EXPERIMENT_ID,
            'backbone': backbone,
            'epoch': epoch,
            'train_loss': train_metrics['loss'],
            'train_accuracy': train_metrics['accuracy'],
            'train_macro_f1': train_metrics['macro_f1'],
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
        }
        history.append(row)
        all_histories.append(row)
        print(row)
        if val_metrics['macro_f1'] > best_val_f1:
            best_val_f1 = val_metrics['macro_f1']
            torch.save({
                'experiment_id': EXPERIMENT_ID,
                'backbone': backbone,
                'model_state_dict': model.state_dict(),
                'label_names': label_names,
                'label_to_id': label_to_id,
                'selected_split': selected_name,
                'img_size': IMG_SIZE,
                'created_at_utc': datetime.now(timezone.utc).isoformat(),
            }, best_path)
            if best_model_info is None or best_val_f1 > best_model_info['val_macro_f1']:
                best_model_info = {'backbone': backbone, 'path': best_path, 'val_macro_f1': best_val_f1}

history_df = pd.DataFrame(all_histories)
history_csv = DRIVE_RUN_ROOT / f'{EXPERIMENT_ID.lower()}_training_history.csv'
history_df.to_csv(history_csv, index=False)
print('history_csv:', history_csv)
print('best_model_info:', best_model_info)


=== Training mobilenet_v3_large ===
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 236MB/s]


  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

{'experiment_id': 'VATTR-EXP-001', 'backbone': 'mobilenet_v3_large', 'epoch': 1, 'train_loss': 1.5893090438842774, 'train_accuracy': 0.37, 'train_macro_f1': 0.3402044716069305, 'val_loss': 1.7975099859843768, 'val_accuracy': 0.3780804150453956, 'val_macro_f1': 0.13113280451363904}


...

  0%|          | 0/25 [00:00<?, ?it/s]

{'experiment_id': 'VATTR-EXP-001', 'backbone': 'mobilenet_v3_large', 'epoch': 2, 'train_loss': 1.3045229387283326, 'train_accuracy': 0.490625, 'train_macro_f1': 0.4670700258577604, 'val_loss': 1.8861713158945166, 'val_accuracy': 0.3780804150453956, 'val_macro_f1': 0.16696853175973128}


  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

{'experiment_id': 'VATTR-EXP-001', 'backbone': 'mobilenet_v3_large', 'epoch': 3, 'train_loss': 1.0826131796836853, 'train_accuracy': 0.565625, 'train_macro_f1': 0.5709332013813331, 'val_loss': 1.956898407868065, 'val_accuracy': 0.33073929961089493, 'val_macro_f1': 0.1900876072618133}
history_csv: /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_attribute/VATTR-EXP-001/vattr-exp-001_training_history.csv
best_model_info: {'backbone': 'mobilenet_v3_large', 'path': PosixPath('/content/anomali-road-safety-ai-work/VATTR-EXP-001/runs/VATTR-EXP-001-mobilenet_v3_large-best.pth'), 'val_macro_f1': 0.1900876072618133}


In [8]:
# Cell 8 - Test best model and export artifacts to Drive
if best_model_info is None:
    raise RuntimeError('No trained model is available. Run the training cell first.')

checkpoint = torch.load(best_model_info['path'], map_location=device)
model = create_model(checkpoint['backbone'], len(checkpoint['label_names'])).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

test_metrics = run_epoch(model, test_loader, None)
report_dict = classification_report(test_metrics['labels'], test_metrics['preds'], target_names=label_names, output_dict=True, zero_division=0)
report_text = classification_report(test_metrics['labels'], test_metrics['preds'], target_names=label_names, zero_division=0)
cm = confusion_matrix(test_metrics['labels'], test_metrics['preds'], labels=list(range(num_classes)))

print('Test metrics:', {k: v for k, v in test_metrics.items() if k not in ['preds', 'labels']})
print(report_text[:4000])

best_drive_path = DRIVE_MODEL_ROOT / f'{EXPERIMENT_ID}-{checkpoint["backbone"]}-best.pth'
shutil.copy2(best_model_info['path'], best_drive_path)

label_map_path = DRIVE_MODEL_ROOT / f'{EXPERIMENT_ID}-label-map.json'
label_map_path.write_text(json.dumps({
    'experiment_id': EXPERIMENT_ID,
    'selected_split': selected_name,
    'label_names': label_names,
    'label_to_id': label_to_id,
}, ensure_ascii=False, indent=2))

metrics_path = DRIVE_RUN_ROOT / f'{EXPERIMENT_ID.lower()}_metrics.json'
metrics_payload = {
    'experiment_id': EXPERIMENT_ID,
    'selected_split': selected_name,
    'smoke_mode': SMOKE_MODE,
    'best_backbone': checkpoint['backbone'],
    'best_checkpoint_drive_path': str(best_drive_path),
    'train_images': int(len(train_df)),
    'val_images': int(len(val_df)),
    'test_images': int(len(test_df)),
    'num_classes': int(num_classes),
    'test_loss': test_metrics['loss'],
    'test_accuracy': test_metrics['accuracy'],
    'test_macro_f1': test_metrics['macro_f1'],
    'classification_report': report_dict,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
}
metrics_path.write_text(json.dumps(metrics_payload, ensure_ascii=False, indent=2))

cm_path = DRIVE_RUN_ROOT / f'{EXPERIMENT_ID.lower()}_confusion_matrix.csv'
pd.DataFrame(cm, index=label_names, columns=label_names).to_csv(cm_path)

print('best_drive_path:', best_drive_path)
print('label_map_path:', label_map_path)
print('metrics_path:', metrics_path)
print('cm_path:', cm_path)

  0%|          | 0/25 [00:00<?, ?it/s]

Test metrics: {'loss': 1.7152908039093018, 'accuracy': 0.260625, 'macro_f1': 0.13388513032631685}
              precision    recall  f1-score   support

       combi       0.56      0.11      0.18       722
   hatchback       0.47      0.40      0.43       584
         mpv       0.00      0.00      0.00        14
       sedan       0.11      0.60      0.19       174
         suv       0.00      0.00      0.00        42
         van       0.00      0.00      0.00        64

    accuracy                           0.26      1600
   macro avg       0.19      0.19      0.13      1600
weighted avg       0.44      0.26      0.26      1600

best_drive_path: /content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/vehicle_attribute/VATTR-EXP-001/VATTR-EXP-001-mobilenet_v3_large-best.pth
label_map_path: /content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/vehicle_attribute/VATTR-EXP-001/VATTR-EXP-001-label-map.json
metrics_path: /content/drive/MyDrive/anomali-road-safety-ai/r

In [9]:
# Cell 9 - Dimension prior table for Speed Fusion Layer
# These are approximate priors. Final values must be calibrated against local data or reliable spec tables.
DIMENSION_PRIOR_RULES = [
    ('motorcycle', {'body_type': 'motorcycle', 'wheelbase_m_mean': 1.40, 'wheelbase_m_min': 1.25, 'wheelbase_m_max': 1.65, 'length_m_mean': 2.05, 'source': 'generic_vehicle_dimension_prior'}),
    ('bus', {'body_type': 'bus', 'wheelbase_m_mean': 6.00, 'wheelbase_m_min': 4.80, 'wheelbase_m_max': 7.50, 'length_m_mean': 11.5, 'source': 'generic_vehicle_dimension_prior'}),
    ('truck', {'body_type': 'truck', 'wheelbase_m_mean': 4.80, 'wheelbase_m_min': 3.40, 'wheelbase_m_max': 6.50, 'length_m_mean': 7.0, 'source': 'generic_vehicle_dimension_prior'}),
    ('van', {'body_type': 'van', 'wheelbase_m_mean': 3.20, 'wheelbase_m_min': 2.75, 'wheelbase_m_max': 3.75, 'length_m_mean': 4.9, 'source': 'generic_vehicle_dimension_prior'}),
    ('mpv', {'body_type': 'mpv', 'wheelbase_m_mean': 2.90, 'wheelbase_m_min': 2.60, 'wheelbase_m_max': 3.25, 'length_m_mean': 4.65, 'source': 'generic_vehicle_dimension_prior'}),
    ('suv', {'body_type': 'suv', 'wheelbase_m_mean': 2.85, 'wheelbase_m_min': 2.60, 'wheelbase_m_max': 3.10, 'length_m_mean': 4.65, 'source': 'generic_vehicle_dimension_prior'}),
    ('hatchback', {'body_type': 'hatchback', 'wheelbase_m_mean': 2.55, 'wheelbase_m_min': 2.35, 'wheelbase_m_max': 2.75, 'length_m_mean': 4.10, 'source': 'generic_vehicle_dimension_prior'}),
    ('sedan', {'body_type': 'sedan', 'wheelbase_m_mean': 2.75, 'wheelbase_m_min': 2.55, 'wheelbase_m_max': 3.00, 'length_m_mean': 4.55, 'source': 'generic_vehicle_dimension_prior'}),
    ('combi', {'body_type': 'wagon', 'wheelbase_m_mean': 2.80, 'wheelbase_m_min': 2.55, 'wheelbase_m_max': 3.05, 'length_m_mean': 4.70, 'source': 'generic_vehicle_dimension_prior'}),
    ('wagon', {'body_type': 'wagon', 'wheelbase_m_mean': 2.80, 'wheelbase_m_min': 2.55, 'wheelbase_m_max': 3.05, 'length_m_mean': 4.70, 'source': 'generic_vehicle_dimension_prior'}),
    ('coupe', {'body_type': 'coupe', 'wheelbase_m_mean': 2.65, 'wheelbase_m_min': 2.40, 'wheelbase_m_max': 2.90, 'length_m_mean': 4.40, 'source': 'generic_vehicle_dimension_prior'}),
]
DEFAULT_PRIOR = {'body_type': 'unknown_car_like', 'wheelbase_m_mean': 2.70, 'wheelbase_m_min': 2.35, 'wheelbase_m_max': 3.05, 'length_m_mean': 4.50, 'source': 'generic_default_prior'}


def prior_for_label(label):
    s = str(label).lower().replace('_', ' ').replace('-', ' ')
    for token, prior in DIMENSION_PRIOR_RULES:
        if token in s:
            return dict(prior)
    # If the selected split is not body-type, the prior is intentionally conservative.
    return dict(DEFAULT_PRIOR)

prior_table = {}
for label in label_names:
    prior = prior_for_label(label)
    prior['label_name'] = label
    prior['selected_split'] = selected_name
    prior['use_for_speed_fusion'] = bool(prior['body_type'] != 'unknown_car_like' or 'body' in selected_name.lower())
    prior_table[label] = prior

prior_path = DRIVE_MODEL_ROOT / f'{EXPERIMENT_ID}-dimension-prior-table.json'
prior_path.write_text(json.dumps({
    'experiment_id': EXPERIMENT_ID,
    'warning': 'Approximate class-level vehicle dimension priors. Not legal/forensic speed evidence. Calibrate before absolute km/h claims.',
    'selected_split': selected_name,
    'priors': prior_table,
}, ensure_ascii=False, indent=2))
print('prior_path:', prior_path)
print(json.dumps(list(prior_table.items())[:5], ensure_ascii=False, indent=2))

prior_path: /content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/vehicle_attribute/VATTR-EXP-001/VATTR-EXP-001-dimension-prior-table.json
[
  [
    "combi",
    {
      "body_type": "wagon",
      "wheelbase_m_mean": 2.8,
      "wheelbase_m_min": 2.55,
      "wheelbase_m_max": 3.05,
      "length_m_mean": 4.7,
      "source": "generic_vehicle_dimension_prior",
      "label_name": "combi",
      "selected_split": "body",
      "use_for_speed_fusion": true
    }
  ],
  [
    "hatchback",
    {
      "body_type": "hatchback",
      "wheelbase_m_mean": 2.55,
      "wheelbase_m_min": 2.35,
      "wheelbase_m_max": 2.75,
      "length_m_mean": 4.1,
      "source": "generic_vehicle_dimension_prior",
      "label_name": "hatchback",
      "selected_split": "body",
      "use_for_speed_fusion": true
    }
  ],
  [
    "mpv",
    {
      "body_type": "mpv",
      "wheelbase_m_mean": 2.9,
      "wheelbase_m_min": 2.6,
      "wheelbase_m_max": 3.25,
      "length_m_mean": 4.65,
      "

In [10]:
# Cell 10 - Optional crop-folder smoke inference
# This does not extract vehicles from video. It only classifies existing crop images if present.

def load_image_for_inference(path):
    image = Image.open(path).convert('RGB')
    return eval_tf(image).unsqueeze(0).to(device)

smoke_rows = []
for crop_dir in SMOKE_CROP_DIRS:
    crop_dir = Path(crop_dir)
    if not crop_dir.exists():
        print('Smoke crop dir missing, skipping:', crop_dir)
        continue
    image_paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        image_paths.extend(crop_dir.rglob(ext))
    image_paths = image_paths[:100]
    print('Smoke inference images:', crop_dir, len(image_paths))
    for image_path in tqdm(image_paths, desc=f'smoke {crop_dir.name}'):
        with torch.no_grad():
            logits = model(load_image_for_inference(image_path))
            prob = torch.softmax(logits, dim=1)[0]
            conf, pred_id = torch.max(prob, dim=0)
        label = label_names[int(pred_id.item())]
        prior = prior_table[label]
        smoke_rows.append({
            'image_path': str(image_path),
            'pred_label': label,
            'confidence': float(conf.item()),
            'body_type_prior': prior['body_type'],
            'wheelbase_m_mean': prior['wheelbase_m_mean'],
            'use_for_speed_fusion': prior['use_for_speed_fusion'],
        })

smoke_df = pd.DataFrame(smoke_rows)
if not smoke_df.empty:
    smoke_csv = DRIVE_RUN_ROOT / f'{EXPERIMENT_ID.lower()}_crop_smoke_predictions.csv'
    smoke_df.to_csv(smoke_csv, index=False)
    print('smoke_csv:', smoke_csv)
    display(smoke_df.head(20))
else:
    print('No smoke crop predictions produced.')

Smoke crop dir missing, skipping: /content/drive/MyDrive/anomali-road-safety-ai/runs/plate_ocr/POCR-EXP-001-target-roi-crops
No smoke crop predictions produced.


In [11]:
# Cell 11 - Write report-ready Markdown summary
summary_md = DRIVE_REPORT_ROOT / f'{EXPERIMENT_ID.lower()}_summary.md'

best_checkpoint_path = str(best_drive_path) if 'best_drive_path' in globals() else 'not trained'
metrics_summary = metrics_payload if 'metrics_payload' in globals() else {}

summary = f"""# {EXPERIMENT_ID} BoxCars Vehicle Attribute / Dimension Prior Classifier

## Amaç

Araç crop'larından hız kestirimi için kullanılabilecek araç nitelik sinyalleri üretmek:

- araç sınıfı / gövde tipi veya fine-grained etiket,
- yaklaşık boyut ön bilgisi,
- `Speed Fusion Layer` için kullanılabilirlik bayrağı.

Bu modül doğrudan hız ölçümü değildir. Hız modelinde plate-scale, homography/track ve dimension-prior sinyallerinin birleştirilmesi gerekir.

## Dataset

- Kaynak: BoxCars116k
- Seçilen split: `{selected_name}`
- Train/Val/Test image sayısı: `{len(train_df)}` / `{len(val_df)}` / `{len(test_df)}`
- Sınıf sayısı: `{num_classes}`
- Smoke mode: `{SMOKE_MODE}`

## Model

- En iyi backbone: `{metrics_summary.get('best_backbone', 'n/a')}`
- Checkpoint: `{best_checkpoint_path}`
- Label map: `{label_map_path if 'label_map_path' in globals() else 'n/a'}`
- Dimension prior table: `{prior_path if 'prior_path' in globals() else 'n/a'}`

## Test Sonucu

- Accuracy: `{metrics_summary.get('test_accuracy', 'n/a')}`
- Macro-F1: `{metrics_summary.get('test_macro_f1', 'n/a')}`

## Speed Fusion Contract

Önerilen runtime çıktısı:

```json
{{
  "track_id": "TRK-17",
  "vehicle_attribute": {{
    "label": "sedan_or_fine_grained_class",
    "confidence": 0.82,
    "body_type_prior": "sedan",
    "wheelbase_m_mean": 2.75,
    "wheelbase_m_min": 2.55,
    "wheelbase_m_max": 3.00,
    "use_for_speed_fusion": true
  }}
}}
```

## Uyarılar

- Marka/model veya gövde tipi tahmini yanlışsa wheelbase prior hız hesabını bozabilir.
- Bu nedenle `use_for_speed_fusion=false` olan veya düşük güvenli tahminler mutlak km/s hesabında kullanılmamalıdır.
- Boyut prior değerleri yaklaşık sınıf ön bilgisidir; yerel kalibrasyon veya güvenilir araç teknik ölçü tablosu ile iyileştirilmelidir.

## Sonraki Adım

1. `KPT-EXP-001`: OpenPifPaf ApolloCar3D pretrained keypoint smoke test.
2. `SPEED-EXP-004`: Plate-scale + homography/track + vehicle dimension prior fusion.
"""
summary_md.write_text(summary)
print('summary_md:', summary_md)
print(summary)

summary_md: /content/drive/MyDrive/anomali-road-safety-ai/reports/vehicle_attribute/VATTR-EXP-001/vattr-exp-001_summary.md
# VATTR-EXP-001 BoxCars Vehicle Attribute / Dimension Prior Classifier

## Amaç

Araç crop'larından hız kestirimi için kullanılabilecek araç nitelik sinyalleri üretmek:

- araç sınıfı / gövde tipi veya fine-grained etiket,
- yaklaşık boyut ön bilgisi,
- `Speed Fusion Layer` için kullanılabilirlik bayrağı.

Bu modül doğrudan hız ölçümü değildir. Hız modelinde plate-scale, homography/track ve dimension-prior sinyallerinin birleştirilmesi gerekir.

## Dataset

- Kaynak: BoxCars116k
- Seçilen split: `body`
- Train/Val/Test image sayısı: `1600` / `1542` / `1600`
- Sınıf sayısı: `6`
- Smoke mode: `True`

## Model

- En iyi backbone: `mobilenet_v3_large`
- Checkpoint: `/content/drive/MyDrive/anomali-road-safety-ai/models/checkpoints/vehicle_attribute/VATTR-EXP-001/VATTR-EXP-001-mobilenet_v3_large-best.pth`
- Label map: `/content/drive/MyDrive/anomali-road-safety-ai/models

## Beklenen Sonraki Adım

1. `VATTR-EXP-001` smoke run başarılıysa ağır run açılır.
2. Ağır run sonrası checkpoint Drive'dan local repo runtime model klasörüne bilinçli olarak taşınır; `.pth` dosyası Git'e eklenmez.
3. `KPT-EXP-001` ile OpenPifPaf ApolloCar3D pretrained keypoint smoke test yapılır.
4. `Speed Fusion Layer` üç sinyali birleştirir:
   - plate-scale speed candidate,
   - homography/track speed candidate,
   - vehicle dimension / wheelbase prior candidate.